# 14. Blessed 90M off-curve runs (baseline / Full / Block × 3 seeds)

Goal: run the **90M off-curve config** for all three architectures at **~459M tokens each** (~5 tokens/param), with **inline validation loss**, **per-layer magnitude/gradient logging**, and **throughput logging**. Real results run — **no downstream benchmark eval during training** (benchmarks run separately on saved checkpoints afterward).

- **Model**: ~91.9M params (`d_model=640`, `n_layers=12`, `n_heads=10`, `d_ff=2560`)
- **Hyperparameters**: `configs/fineweb_90m_offcurve.yaml` (AdamW 0.9/0.95, wd 0.1, eps 1e-8, cosine + warmup, bf16, effective batch 65,536 tokens)
- **Variants × seeds**: `baseline`, `attnres` (Full), `block_attnres` (Block, `num_blocks=6` → 2 layers/block) × seeds `{123, 456, 1337}` = **9 runs**
- **Block note**: 12 layers / 6 blocks gives clearer sawtooth geometry than the 30M (`num_blocks=3`) setup — better Fig. 5b signal
- **Dataset**: `fineweb_edu` sample-10BT, `block_size=1024`, existing hash split
- **Logging**: train loss every 10 steps, **val loss every 100 steps**, per-layer `h_l` + `dL/dh_l`, `tokens_per_sec` + wall-clock, checkpoints + summaries to **Google Drive**

Flow: sync repo → mount Drive → GPU check → pre-launch report (config / 9-run plan / cost) → VRAM probe → **confirmation gate** → launch sequentially.

> **Note:** Notebook 13 is the blessed 30M two-seed suite. This notebook is the 90M off-curve follow-on.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# W&B: enabled with mode=auto (online if WANDB_API_KEY set, else offline).
# Set via Colab secrets / env — do not hardcode keys in the notebook.
# os.environ["WANDB_API_KEY"] = "<your-key>"

print("repo:", REPO)
print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)
print("artifacts layout:")
print("  checkpoints/", DRIVE_ROOT / "checkpoints")
print("  runs/       ", DRIVE_ROOT / "runs")
print("  logs/       ", DRIVE_ROOT / "logs")

In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device_name =", props.name)
    print("total_vram_gib =", round(props.total_memory / (1024 ** 3), 1))
    print("bf16_supported =", torch.cuda.is_bf16_supported())

## Pre-launch report: 90M config + 9-run plan + logging checks + cost estimate

Review this cell's output before enabling the confirmation gate.

In [ ]:
from src.utils.config import load_config
from src.models.attnres import build_model
from src.utils.logging import build_run_identity
from src.utils.runtime import count_parameters

CONFIG_PATH = "configs/fineweb_90m_offcurve.yaml"
EFFECTIVE_BATCH_TOKENS = 65_536
SEEDS = [123, 456, 1337]

RUN_PLAN = [
    {
        "label": "baseline",
        "architecture": "baseline",
        "overrides": ["model.architecture=baseline", "model.attnres.enabled=false"],
        "notes": "Standard PreNorm residual GPT.",
    },
    {
        "label": "full_attnres",
        "architecture": "attnres",
        "overrides": [
            "model.architecture=attnres",
            "model.attnres.enabled=true",
            "model.attnres.mode=full",
        ],
        "notes": "Full depth-attention over all prior sublayer outputs.",
    },
    {
        "label": "block_attnres",
        "architecture": "block_attnres",
        "overrides": ["model.architecture=block_attnres"],
        "notes": "Block AttnRes (num_blocks=6 from YAML → 2 layers/block).",
    },
]

cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}"])
eff_tokens = cfg.data.batch_size * cfg.training.grad_accum_steps * cfg.data.block_size
assert eff_tokens == EFFECTIVE_BATCH_TOKENS, f"effective batch is {eff_tokens}, expected {EFFECTIVE_BATCH_TOKENS}"
assert cfg.training.eval_interval > 0, "eval_interval must be > 0 for inline val loss"
assert cfg.evaluation.positionwise_steps == [], "no positionwise / benchmark steps during training"
assert cfg.model.attnres.num_blocks == 6, "Block AttnRes expects num_blocks=6 in YAML"

param_counts = {}
for plan in RUN_PLAN:
    variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}", *plan["overrides"]])
    param_counts[plan["label"]] = count_parameters(build_model(variant_cfg.model))["total"]

run_names = {}
for seed in SEEDS:
    for plan in RUN_PLAN:
        variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={seed}", *plan["overrides"]])
        run_names[(seed, plan["label"])] = build_run_identity(variant_cfg).run_name

tokens_per_run = cfg.training.max_steps * EFFECTIVE_BATCH_TOKENS
tokens_total = len(SEEDS) * len(RUN_PLAN) * tokens_per_run

# Wall-clock estimate: scale 30M A100 timings by (params × tokens), with a Full AttnRes
# depth-attn bump for L=12 (depth-attn cost grows faster than dense FLOPs).
WALL_30M_HOURS = {"baseline": 0.94, "full_attnres": 1.00, "block_attnres": 0.80}
PARAMS_30M = 30_334_464
TOKENS_30M = 150_011_904
flops_scale = (param_counts["baseline"] / PARAMS_30M) * (tokens_per_run / TOKENS_30M)
FULL_DEPTH_ATTN_BUMP = 1.25
est_hours = {
    "baseline": WALL_30M_HOURS["baseline"] * flops_scale,
    "full_attnres": WALL_30M_HOURS["full_attnres"] * flops_scale * FULL_DEPTH_ATTN_BUMP,
    "block_attnres": WALL_30M_HOURS["block_attnres"] * flops_scale,
}
suite_hours_est = sum(est_hours[p["label"]] for p in RUN_PLAN for _ in SEEDS)
A100_USD_PER_HOUR = 2.0  # Colab Pro+/typical on-demand ballpark; adjust if needed

print("=== 90M off-curve config ===")
print(f"config file     : {CONFIG_PATH}")
print(f"model shape     : d_model={cfg.model.d_model} n_layers={cfg.model.n_layers} "
      f"n_heads={cfg.model.n_heads} d_ff={cfg.model.d_ff}")
print(f"param counts    : baseline={param_counts['baseline']:,}  "
      f"full={param_counts['full_attnres']:,}  block={param_counts['block_attnres']:,}")
print(f"num_blocks YAML : {cfg.model.attnres.num_blocks}  (Block: 2 transformer layers/block)")
print(f"dataset         : {cfg.data.dataset_name} / {cfg.data.fineweb_subset}  "
      f"block_size={cfg.data.block_size}  hash_modulo={cfg.data.hash_modulo}  "
      f"val_fraction={cfg.data.val_fraction}")
print(f"optimizer       : AdamW lr={cfg.training.learning_rate} min_lr={cfg.training.min_lr} "
      f"betas=({cfg.training.beta1},{cfg.training.beta2}) wd={cfg.training.weight_decay} "
      f"eps={cfg.training.adam_eps} clip={cfg.training.grad_clip}")
print(f"schedule        : cosine  warmup={cfg.training.warmup_steps}  "
      f"max_steps={cfg.training.max_steps}  amp={cfg.training.amp_dtype}")
print(f"YAML microbatch : {cfg.data.batch_size} x {cfg.training.grad_accum_steps} accum "
      f"x {cfg.data.block_size} ctx = {eff_tokens:,} tokens/step")
print(f"horizon/run     : {tokens_per_run:,} tokens ({tokens_per_run/1e6:.1f}M)  "
      f"~{tokens_per_run / param_counts['baseline']:.2f} tokens/param")
print(f"total budget    : {tokens_total:,} tokens ({tokens_total/1e6:.1f}M) across 9 runs")
print(f"ckpt interval   : every {cfg.training.checkpoint_interval} steps "
      f"({cfg.training.checkpoint_interval * EFFECTIVE_BATCH_TOKENS / 1e6:.1f}M tokens)")

print("\n=== logging wiring checks ===")
print(f"  train loss        : log_interval={cfg.training.log_interval}  ✓")
print(f"  val loss (inline) : eval_interval={cfg.training.eval_interval}  "
      f"eval_max_batches={cfg.training.eval_max_batches}  ✓")
print("  per-layer |h_l|   : LayerInputMagnitudeTracker on every train step + final eval  ✓")
print("  per-layer |dL/dh| : same tracker (gradient hooks) logged with train metrics  ✓")
print("  throughput        : tokens_per_sec, step_time_sec, elapsed_time_sec every log  ✓")
print(f"  benchmarks in-train: positionwise_steps={cfg.evaluation.positionwise_steps}  "
      f"(empty = off)  ✓")
print(f"  Drive artifacts   : checkpoints/ runs/ logs/ under {DRIVE_ROOT}")

print("\n=== 9-run plan ===")
for seed in SEEDS:
    print(f"seed={seed}")
    for plan in RUN_PLAN:
        print(f"  {plan['label']:14s} -> {run_names[(seed, plan['label'])]}")

print("\n=== A100 cost / wall-clock estimate ===")
print(f"scale from 30M A100 timings: flops_scale={flops_scale:.2f}x  "
      f"(+{FULL_DEPTH_ATTN_BUMP:.0%} Full depth-attn bump)")
for label, hours in est_hours.items():
    print(f"  {label:14s}: ~{hours:.1f} h / run  × 3 seeds = ~{3 * hours:.1f} h")
print(f"  TOTAL suite     : ~{suite_hours_est:.0f} h wall-clock (sequential)")
print(f"  @ ${A100_USD_PER_HOUR:.1f}/h A100 : ~${suite_hours_est * A100_USD_PER_HOUR:.0f}  "
      f"(ballpark; Colab Pro+ / on-demand rates vary ~$1.5–$3/h → "
      f"~${suite_hours_est * 1.5:.0f}–${suite_hours_est * 3.0:.0f})")
print("\nSTOP: review this report, run the VRAM probe, then set CONFIRM_LAUNCH=True.")

## VRAM probe: shared micro-batch for all variants

Identical hyperparameters across variants — one shared micro-batch. Probes **Full AttnRes** (worst case at L=12) and picks the largest micro-batch that fits while keeping effective batch = 65,536 tokens. YAML default is `8 × 8`; set `FORCE_MICRO_BATCH` to skip the probe.

In [ ]:
import gc

EFFECTIVE_BATCH_SEQUENCES = EFFECTIVE_BATCH_TOKENS // cfg.data.block_size  # 64
VRAM_HEADROOM = 0.85
FORCE_MICRO_BATCH = None  # e.g. 8 to skip probe and use YAML default
CANDIDATES = [32, 16, 8, 4, 2, 1]


def _probe_fits(architecture: str, micro_batch: int, overrides: list[str]) -> bool:
    if not torch.cuda.is_available():
        return True
    torch.cuda.empty_cache()
    gc.collect()
    probe_cfg = load_config(
        CONFIG_PATH,
        overrides=[
            f"experiment.seed={SEEDS[0]}",
            f"data.batch_size={micro_batch}",
            *overrides,
        ],
    )
    model = build_model(probe_cfg.model).cuda()
    model.train()
    x = torch.randint(0, probe_cfg.model.vocab_size, (micro_batch, probe_cfg.data.block_size), device="cuda")
    y = torch.randint(0, probe_cfg.model.vocab_size, (micro_batch, probe_cfg.data.block_size), device="cuda")
    try:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits, _ = model(x, return_aux=False)
            loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
        budget = VRAM_HEADROOM * (torch.cuda.get_device_properties(0).total_memory / (1024 ** 2))
        ok = peak < budget
        print(f"  {architecture:14s} mb={micro_batch:2d}  peak={peak:,.0f} MiB  "
              f"budget={budget:,.0f}  {'OK' if ok else 'OOM-risk'}")
        return ok
    except torch.cuda.OutOfMemoryError:
        print(f"  {architecture:14s} mb={micro_batch:2d}  OOM")
        return False
    finally:
        del model
        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.reset_peak_memory_stats()


if FORCE_MICRO_BATCH is not None:
    if EFFECTIVE_BATCH_SEQUENCES % FORCE_MICRO_BATCH != 0:
        raise ValueError(f"FORCE_MICRO_BATCH={FORCE_MICRO_BATCH} must divide {EFFECTIVE_BATCH_SEQUENCES}")
    MICRO_BATCH = FORCE_MICRO_BATCH
    GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // FORCE_MICRO_BATCH
    print(f"FORCED micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH * GRAD_ACCUM * cfg.data.block_size:,} tokens/step (probe skipped)")
else:
    print("Probing Full AttnRes (worst case) for shared micro-batch…")
    chosen = None
    full_overrides = next(p["overrides"] for p in RUN_PLAN if p["label"] == "full_attnres")
    for mb in CANDIDATES:
        if EFFECTIVE_BATCH_SEQUENCES % mb != 0:
            continue
        if _probe_fits("full_attnres", mb, full_overrides):
            chosen = mb
            break
    if chosen is None:
        raise RuntimeError("No micro-batch fit Full AttnRes under VRAM headroom")
    MICRO_BATCH = chosen
    GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // chosen
    print(f"SELECTED micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH * GRAD_ACCUM * cfg.data.block_size:,} tokens/step (shared by all variants)")

## Confirmation gate (must be True to launch)

Set `CONFIRM_LAUNCH = True` only after reviewing the pre-launch report and VRAM probe. Leave `CLEAN_EXISTING_RUNS = False` unless you intentionally want to wipe Drive artifacts for these 9 run names before a fresh start.

In [ ]:
import shutil
from src.training.train import train_from_config

CONFIRM_LAUNCH = False
CLEAN_EXISTING_RUNS = False

SHARED_OVERRIDES = [
    f"logging.output_root={DRIVE_ROOT}",
    f"data.batch_size={MICRO_BATCH}",
    f"data.eval_batch_size={MICRO_BATCH}",
    f"training.grad_accum_steps={GRAD_ACCUM}",
    "logging.wandb.group=blessed_90m_offcurve",
]

print("launch plan:")
print("  CONFIRM_LAUNCH      =", CONFIRM_LAUNCH)
print("  CLEAN_EXISTING_RUNS =", CLEAN_EXISTING_RUNS)
print(f"  micro_batch/grad_accum = {MICRO_BATCH} / {GRAD_ACCUM}")
print(f"  est suite wall-clock   ≈ {suite_hours_est:.0f} h  "
      f"(~${suite_hours_est * A100_USD_PER_HOUR:.0f} @ ${A100_USD_PER_HOUR:.1f}/h)")
for seed in SEEDS:
    for plan in RUN_PLAN:
        print(
            f"  seed={seed}  {plan['label']:14s}  "
            f"est≈{est_hours[plan['label']]:.1f}h  run={run_names[(seed, plan['label'])]}"
        )

if not CONFIRM_LAUNCH:
    raise RuntimeError(
        "Set CONFIRM_LAUNCH=True after reviewing the 90M report / cost estimate / VRAM probe, then rerun."
    )

RUN_SUMMARIES = []
suite_start = time.time()

for seed in SEEDS:
    for plan in RUN_PLAN:
        label = plan["label"]
        run_name = run_names[(seed, label)]

        print("\n" + "=" * 72)
        print(f"LAUNCHING seed={seed} label={label} arch={plan['architecture']} "
              f"mb={MICRO_BATCH} ga={GRAD_ACCUM}")
        print("=" * 72)

        if CLEAN_EXISTING_RUNS:
            for sub in ("runs", "checkpoints"):
                path = DRIVE_ROOT / sub / run_name
                if path.exists():
                    print("removing", path)
                    shutil.rmtree(path)

        overrides = SHARED_OVERRIDES + [
            f"experiment.seed={seed}",
            *plan["overrides"],
            f"logging.wandb.run_name={run_name}",
        ]
        launch_cfg = load_config(CONFIG_PATH, overrides=overrides)
        assert launch_cfg.training.eval_interval > 0
        assert (
            launch_cfg.data.batch_size
            * launch_cfg.training.grad_accum_steps
            * launch_cfg.data.block_size
            == EFFECTIVE_BATCH_TOKENS
        )
        if label == "block_attnres":
            assert launch_cfg.model.attnres.num_blocks == 6

        run_start = time.time()
        summary = train_from_config(launch_cfg)
        run_hours = (time.time() - run_start) / 3600

        RUN_SUMMARIES.append({
            "seed": seed,
            "label": label,
            "micro_batch": MICRO_BATCH,
            "grad_accum": GRAD_ACCUM,
            "run_name": summary.get("run_name"),
            "val_loss": summary.get("val_loss"),
            "best_val_loss": summary.get("best_val_loss"),
            "tokens_seen": summary.get("tokens_seen"),
            "wall_clock_hours": run_hours,
            "checkpoint_path": summary.get("checkpoint_path"),
            "wandb_url": summary.get("wandb_url"),
            "mean_layer_input_magnitude_last_layer": summary.get(
                "mean_layer_input_magnitude_last_layer"
            ),
        })

        print(f"Finished seed={seed} label={label} in {run_hours:.2f} h")
        print("  val_loss      :", summary.get("val_loss"))
        print("  best_val_loss :", summary.get("best_val_loss"))
        print("  last |h_L|    :", summary.get("mean_layer_input_magnitude_last_layer"))
        print("  checkpoint    :", summary.get("checkpoint_path"))

_TRAINING_COMPLETED = True
suite_hours = (time.time() - suite_start) / 3600
print("\n" + "=" * 72)
print(f"ALL RUNS COMPLETE in {suite_hours:.2f} h")
for row in RUN_SUMMARIES:
    print(
        f"seed={row['seed']}  {row['label']:14s}  "
        f"val={row['val_loss']}  best={row['best_val_loss']}  wall={row['wall_clock_hours']:.2f}h"
    )

## End the Colab session on completion

Unassigns the runtime after all runs finish so idle compute units aren't burned.

In [ ]:
UNASSIGN_ON_COMPLETE = True

training_completed = globals().get("_TRAINING_COMPLETED", False)

if UNASSIGN_ON_COMPLETE and training_completed:
    print("Training complete - ending Colab session to free the A100.")
    from google.colab import runtime
    runtime.unassign()
else:
    print(
        f"Not unassigning (UNASSIGN_ON_COMPLETE={UNASSIGN_ON_COMPLETE}, "
        f"training_completed={training_completed})."
    )